# 01 - Ingestão Bronze
Preserva o texto original do COTAHIST e a referência setorial com metadados de carga.

In [0]:
%run ./00_setup

# 00 - Configuração
Cria objetos do Unity Catalog e caminhos do MVP. Ajuste os widgets antes da primeira execução.

Envie COTAHIST_A2025.TXT para: /Volumes/workspace/mvp_b3/landing/cotahist/
Envie setores_b3.csv para: /Volumes/workspace/mvp_b3/landing/referencia/


In [0]:
from pyspark.sql import functions as F

cotahist_path = f"{base_path}/cotahist/*.TXT"
setores_path = f"{base_path}/referencia/setores_b3.csv"

# Leitura do COTAHIST e preservação do dado bruto
raw = (
    spark.read.text(cotahist_path)
    .select(
        F.col("value").alias("raw_line"),
        F.col("_metadata.file_path").alias("source_file")
    )
    .filter(
        F.substring("raw_line", 1, 2) == "01"
    )
    .withColumn(
        "ingestion_ts",
        F.current_timestamp()
    )
    .withColumn(
        "record_hash",
        F.sha2("raw_line", 256)
    )
)

# Gravação da tabela Bronze do COTAHIST
(
    raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalog}.{schema}.bronze_cotahist_raw"
    )
)

# Leitura da classificação setorial
setores = (
    spark.read
    .option("header", True)
    .option("encoding", "UTF-8")
    .option("sep", ",")
    .option("quote", '"')
    .csv(setores_path)
    .select(
        *[
            F.trim(F.col(coluna)).alias(coluna)
            for coluna in [
                "ticker",
                "empresa",
                "setor",
                "subsetor",
                "segmento"
            ]
        ]
    )
    .withColumn(
        "ingestion_ts",
        F.current_timestamp()
    )
)

# Gravação da tabela Bronze da classificação setorial
(
    setores.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        f"{catalog}.{schema}.bronze_setores_raw"
    )
)

print("Ingestão Bronze concluída com sucesso.")

Ingestão Bronze concluída com sucesso.


In [0]:
display(
    spark.sql(
        f"""
        SELECT
            'cotahist' AS fonte,
            COUNT(*) AS registros
        FROM {catalog}.{schema}.bronze_cotahist_raw

        UNION ALL

        SELECT
            'setores' AS fonte,
            COUNT(*) AS registros
        FROM {catalog}.{schema}.bronze_setores_raw
        """
    )
)

fonte,registros
cotahist,3174698
setores,405
